

# Create a design study from an existing project

This example demonstrates how to create and manage a design study
from an existing optiSLang project.

Instead of generating a new workflow from a template, this example demonstrates
direct instantiation of a parametric design study with existing optiSLang nodes.
The parametric design study is executed then.

<div class="alert alert-info"><h4>Note</h4><p>This example requires a workflow generated by the `ref_rdo_proxy_solver` example.
    Run the workflow creation part of *Robust design with optimization with proxy solver* example first
    and make sure the workflow is created as expected.</p></div>

<img src="file://../../../_static/04_2_RDO_w_proxysolver.png" width="1200" alt="Required workflow.">


## Perform required imports
Perform the required imports.



In [ ]:
from pathlib import Path

from ansys.optislang.core.nodes import IntegrationNode, ParametricSystem, ProxySolverNode
from ansys.optislang.core.project_parametric import Design, DesignVariable
from ansys.optislang.parametric.design_study import (
    ExecutableBlock,
    ExecutionOption,
    ManagedInstance,
    ManagedParametricSystem,
    ParametricDesignStudy,
    ParametricDesignStudyManager,
    ProxySolverManagedParametricSystem,
)

## Define proxy solver callback
Define a simple calculator function to solve the variations.



In [ ]:
def calculator(X1, X2, X3, X4, X5):
    from math import sin, sqrt

    Y = 0.5 * X1 + X2 + 0.5 * X1 * X2 + 5 * sin(X3) + 0.2 * X4 + 0.1 * X5
    Z = ((-1) * sqrt(abs(Y))) ** 3
    return Y, Z


def calculate(designs: list[Design]):
    result_designs = []
    for design in designs:
        X1 = design.parameters[design.parameters_names.index("X1")].value
        X2 = design.parameters[design.parameters_names.index("X2")].value
        X3 = design.parameters[design.parameters_names.index("X3")].value
        X4 = design.parameters[design.parameters_names.index("X4")].value
        X5 = design.parameters[design.parameters_names.index("X5")].value
        Y, Z = calculator(X1, X2, X3, X4, X5)
        design_id = design.id

        # create instance of design with new values
        output_design = Design(
            responses=[DesignVariable("Y", Y), DesignVariable("Z", Z)],
            design_id=design_id,
        )
        result_designs.append(output_design)
    return result_designs

## Specify the project file path
Specify the path to the existing project file generated by the `ref_rdo_proxy_solver`
example. Update the path below to point to the saved project file.



In [ ]:
project_path = Path("rdo_workflow.opf").resolve()
if not project_path.is_file():
    raise FileNotFoundError(
        f"Project file not found: {project_path}. Update 'project_path' to point to the .opf created by the ref_rdo_proxy_solver example."
    )

## Open the project
Create ParametricDesignStudyManager instance and access the Optislang instance.



In [ ]:
manager = ParametricDesignStudyManager(project_path=project_path)
osl = manager.optislang
rs = osl.application.project.root_system

## Access the workflow
Locate systems and nodes to be passed into ParametricDesignStudy constructor.



In [ ]:
managed_instances = []
execution_blocks = []

# AMOP block
# ----------

amop_system: ParametricSystem = rs.find_nodes_by_name("AMOP")[0]
amop_proxy_solver: ProxySolverNode = amop_system.get_nodes()[0]
amop_instance = ProxySolverManagedParametricSystem(
    algorithm=amop_system, solver_node=amop_proxy_solver, callback=calculate
)
amop_block = ExecutableBlock(
    (
        (
            amop_instance,
            ExecutionOption.ACTIVE | ExecutionOption.STARTING_POINT | ExecutionOption.END_POINT,
        ),
    )
)

managed_instances.append(amop_instance)
execution_blocks.append(amop_block)

# OCO MOP block
# -------------

oco_mop: ParametricSystem = rs.find_nodes_by_name("OCO_MOP")[0]
oco_mop_solver: IntegrationNode = oco_mop.get_nodes()[0]
oco_mop_instance = ManagedParametricSystem(parametric_system=oco_mop, solver_node=oco_mop_solver)
oco_mop_block = ExecutableBlock(
    (
        (
            oco_mop_instance,
            ExecutionOption.ACTIVE | ExecutionOption.STARTING_POINT | ExecutionOption.END_POINT,
        ),
    )
)
managed_instances.append(oco_mop_instance)
execution_blocks.append(oco_mop_block)

# Validator block
# ---------------

validator_system: ParametricSystem = rs.find_nodes_by_name("Validator System")[0]
validator_proxy_solver: ProxySolverNode = validator_system.get_nodes()[0]
validator_instance = ProxySolverManagedParametricSystem(
    algorithm=validator_system, solver_node=validator_proxy_solver, callback=calculate
)

filter_node = rs.find_nodes_by_name("VALIDATOR_FILTER_NODE")[0]
filter_node_instance = ManagedInstance(filter_node)
append_designs_node = rs.find_nodes_by_name("Append Designs")[0]
append_designs_node_instance = ManagedInstance(append_designs_node)
postprocessing_node = rs.find_nodes_by_name("PostProcessing")[0]
postprocessing_node_instance = ManagedInstance(postprocessing_node)

validator_block = ExecutableBlock(
    (
        (
            validator_instance,
            ExecutionOption.ACTIVE,
        ),
        (
            filter_node_instance,
            ExecutionOption.ACTIVE | ExecutionOption.STARTING_POINT,
        ),
        (
            append_designs_node_instance,
            ExecutionOption.ACTIVE,
        ),
        (
            postprocessing_node_instance,
            ExecutionOption.ACTIVE | ExecutionOption.END_POINT,
        ),
    )
)

managed_instances.extend(
    [
        validator_instance,
        filter_node_instance,
        append_designs_node_instance,
        postprocessing_node_instance,
    ]
)
execution_blocks.append(validator_block)

# OCO solver block
# ----------------

oco_solver_system: ParametricSystem = rs.find_nodes_by_name("OCO_SOLVER")[0]
oco_solver_proxy_solver: ProxySolverNode = oco_solver_system.get_nodes()[0]
oco_solver_instance = ProxySolverManagedParametricSystem(
    algorithm=oco_solver_system, solver_node=oco_solver_proxy_solver, callback=calculate
)
oco_solver_block = ExecutableBlock(
    (
        (
            oco_solver_instance,
            ExecutionOption.ACTIVE | ExecutionOption.STARTING_POINT | ExecutionOption.END_POINT,
        ),
    )
)

managed_instances.append(oco_solver_instance)
execution_blocks.append(oco_solver_block)

# Robustness block
# ----------------

robustness_system: ParametricSystem = rs.find_nodes_by_name("Robustness")[0]
robustness_proxy_solver: ProxySolverNode = robustness_system.get_nodes()[0]
robustness_instance = ProxySolverManagedParametricSystem(
    algorithm=robustness_system, solver_node=robustness_proxy_solver, callback=calculate
)
robustness_block = ExecutableBlock(
    (
        (
            robustness_instance,
            ExecutionOption.ACTIVE | ExecutionOption.STARTING_POINT | ExecutionOption.END_POINT,
        ),
    )
)

managed_instances.append(robustness_instance)
execution_blocks.append(robustness_block)

# MOP block
# ---------
mop_node: ParametricSystem = rs.find_nodes_by_name("MOP")[0]
mop_instance = ManagedInstance(mop_node)
mop_block = ExecutableBlock(
    (
        (
            mop_instance,
            ExecutionOption.ACTIVE | ExecutionOption.STARTING_POINT | ExecutionOption.END_POINT,
        ),
    )
)

managed_instances.append(mop_instance)
execution_blocks.append(mop_block)

## Create design study
Create design study and append it to the manager.



In [ ]:
design_study = ParametricDesignStudy(
    osl_instance=osl, managed_instances=managed_instances, execution_blocks=execution_blocks
)

manager.append_design_study(design_study=design_study)

## Execute design study
Execute the design study.



In [ ]:
manager.reset()
manager.save()
design_study.execute()
manager.save()
manager.dispose()